# LangChain이 왜 필요한가

LLM을 한 번 호출하는 것은 LangChain 없이도 가능.  
하지만 실제 애플리케이션은 보통 다음과 같은 여러 단계를 거침.

1. 사용자 질문 입력
2. 시스템 프롬프트 적용
3. 관련 문서 검색
4. 검색 결과를 문맥으로 조립
5. 모델 호출
6. 출력 형식 정리
7. 필요하면 도구 호출

이 과정이 길어질수록 코드가 복잡해짐.

LangChain은 이 문제를 해결하기 위해 **모델, 프롬프트, 문서, 검색, 도구, 에이전트** 같은 요소를 서로 연결 가능한 구성요소로 다룰 수 있게 해줌.

# LangChain의 핵심 구성요소

LangChain을 처음 배울 때는 아래 7개를 가장 먼저 이해하면 됨.

## (1) Model
LLM 또는 chat model을 호출하는 인터페이스

## (2) Prompt
입력을 구조화하는 템플릿

## (3) Chain
여러 단계를 연결한 실행 흐름

## (4) Document
문서 단위를 표현하는 객체

## (5) Text Splitter
긴 문서를 작은 chunk로 나누는 도구

## (6) Vector Store / Retriever
문서 조각을 저장하고 관련 내용을 검색하는 도구

## (7) Agent
필요할 때 도구를 선택하고 여러 단계를 수행하는 실행 구조

In [1]:
%pip install -U langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   --------------------------------- ------ 2.1/2.5 MB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 9.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---- ----------------------------------- 2.1/18.9 MB 9.8 MB/s eta 0:00:02
   -------- ------------------------------- 4.2/18.9 MB 10.1 MB/s eta 0:00:02
   ------------- -------------------------- 6.6/18.9 MB 10.1 MB/s eta 0:00:02
   ------------------ --------------------- 8.7/18.9 MB 10.1 MB/s eta 0:00:02
   ---------------------- ----------------- 10.7/18.9 MB 10.2 MB/s eta 0:00:01
   --------------------------- ------------ 13.1/18.9 MB 10.1 MB/s eta 0:00:01
   -------------------------------- ------- 15.2/18.9 MB 10.2 MB/s eta 0:00:01
   ------------------------------------ --- 17.3/18.9 MB 10.2 MB/s eta 0:00:01
   ---------------------------------------  18.9/18.9 MB 10.2 MB/s eta 0:00:01
   

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# python 3.10+ 요구사항이 angchain v1 migration 문서에 명시되어 있음
import sys
print(sys.version)

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


# API 키 설정

OpenAI chat model과 embedding model을 사용.

보안을 위해 API 키를 코드에 직접 쓰지 않고, 입력창을 통해 환경 변수로 설정합니다.

In [3]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

In [4]:
# 확인하는 셀
print("OPENAI_API_KEY" in os.environ)

True


**가장 작은 LangChain 예제**


확인할 것은 두 가지.

1. LangChain으로 chat model 객체를 만들 수 있는가?
2. `invoke()`를 통해 모델을 호출할 수 있는가?

---

## 핵심 개념

- `ChatOpenAI(...)` : 모델 객체 생성
- `invoke(...)` : 실행

In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

response = llm.invoke("LangChain이 무엇인지 3문장으로 설명해줘.")
print(response)

c:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


content='LangChain은 자연어 처리(NLP) 모델을 활용하여 다양한 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다. 이 프레임워크는 데이터 소스와의 통합, 대화형 에이전트 생성, 그리고 복잡한 작업을 수행하는 데 필요한 다양한 도구와 기능을 제공합니다. LangChain을 사용하면 개발자들이 NLP 모델을 보다 쉽게 활용하고, 맞춤형 솔루션을 신속하게 개발할 수 있습니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 21, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_83e2dd34fc', 'id': 'chatcmpl-DVRORYNy8VUoWdcQflokD0oZh0Ult', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d98d8-cff2-7603-afe8-fdf26b425043-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 21, 'output_tokens': 105, 'total_tokens': 126, 'input_token_details': {'audio': 0, 'cach

In [6]:
print(type(response))
print()
print(response.content)

<class 'langchain_core.messages.ai.AIMessage'>

LangChain은 자연어 처리(NLP) 모델을 활용하여 다양한 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다. 이 프레임워크는 데이터 소스와의 통합, 대화형 에이전트 생성, 그리고 복잡한 작업을 수행하는 데 필요한 다양한 도구와 기능을 제공합니다. LangChain을 사용하면 개발자들이 NLP 모델을 보다 쉽게 활용하고, 맞춤형 솔루션을 신속하게 개발할 수 있습니다.


`invoke()`의 결과는 단순 문자열이 아니라 메시지 객체일 수 있다.

LangChain은 단순 텍스트 반환만이 아니라 모델 응답을 구조적으로 다루는 인터페이스를 제공.

**temperature 비교**

같은 질문이라도 temperature 설정에 따라 응답의 안정성과 다양성이 달라질 수 있음.

- temperature = 0
- temperature = 0.7

In [7]:
llm0 = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm7 = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

question = "Transformer를 쉬운 비유로 설명해줘."

r0 = llm0.invoke(question)
r7 = llm7.invoke(question)

print("[temperature=0]")
print(r0.content)
print("\n" + "="*80 + "\n")
print("[temperature=0.7]")
print(r7.content)

[temperature=0]
Transformer를 쉽게 비유하자면, "회의에서 여러 사람의 의견을 듣고 정리하는 과정"이라고 할 수 있습니다.

1. **회의 참석자**: Transformer의 각 단어는 회의에 참석한 사람들입니다. 각 참석자는 자신이 가진 정보를 가지고 있습니다.

2. **의견 교환**: 회의 중에 참석자들은 서로의 의견을 듣고, 어떤 의견이 중요한지 판단합니다. 이 과정에서 각 참석자는 다른 참석자와의 관계를 고려하여 자신의 의견을 조정합니다. Transformer에서는 이 과정을 '어텐션'이라고 부릅니다.

3. **정리**: 회의가 끝나면, 참석자들은 서로의 의견을 바탕으로 최종 결론을 도출합니다. 이처럼 Transformer는 입력된 단어들을 바탕으로 최종적인 출력을 생성합니다.

결국, Transformer는 여러 단어(사람)의 정보를 효과적으로 조합하여 의미 있는 결과(결론)를 만들어내는 시스템이라고 할 수 있습니다.


[temperature=0.7]
Transformer를 비유로 설명하자면, "도서관의 사서"와 비슷하게 생각할 수 있습니다.

1. **도서관의 책**: 도서관에는 다양한 주제의 책들이 있습니다. Transformer는 이 책들을 읽고 이해하여 정보를 처리합니다.

2. **사서의 역할**: 도서관에 사서가 있다고 가정해보세요. 사서는 책을 잘 알고 있어서, 사용자가 원하는 정보를 빠르게 찾고 정리해 줄 수 있습니다. Transformer도 입력된 문장을 잘 이해하고, 그 문장에서 중요한 정보를 파악합니다.

3. **상관관계 파악**: 사서는 책들 간의 관계를 이해하고, 어떤 책이 서로 연결되어 있는지를 알고 있습니다. 마찬가지로, Transformer는 입력된 단어들 간의 관계를 파악하여 문맥을 이해합니다. 이를 통해 어떤 단어가 다른 단어와 어떻게 연결되는지를 분석합니다.

4. **다양한 정보 처리**: 도서관에서는 연구, 공부, 정보 검색 등 다양한 목적으로 책을 찾습니다. Transformer는 다

- 어떤 응답이 더 안정적인가?
- 어떤 응답이 더 표현이 다양한가?
- 강의용 데모에서는 어떤 설정이 더 적절한가?

**PromptTemplate**

프롬프트를 직접 문자열로 만들 수도 있다.

예:
- f-string으로 문자열 조립
- 변수 값을 직접 삽입

하지만 실제 애플리케이션에서는 입력 변수가 많아지고, 재사용 가능성이 중요해짐.

그래서 LangChain은 PromptTemplate를 제공.

---

#### PromptTemplate의 역할

- 프롬프트를 재사용 가능하게 만든다.
- 입력 변수를 명확히 드러낸다.
- 체인으로 연결하기 쉽게 만든다.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(
    """
너는 AI 튜터다.
주제: {topic}
대상: {level}
설명 길이: {n_sentences}문장
쉽고 정확하게 설명해줘.
"""
)

In [9]:
messages = prompt_template.invoke(
    {
        "topic": "RAG",
        "level": "초보자",
        "n_sentences": 4
    }
)

print(messages)
print(type(messages))

messages=[HumanMessage(content='\n너는 AI 튜터다.\n주제: RAG\n대상: 초보자\n설명 길이: 4문장\n쉽고 정확하게 설명해줘.\n', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>


In [10]:
answer = llm.invoke(messages)
print(answer.content)

RAG는 "Retrieval-Augmented Generation"의 약자로, 정보 검색과 생성 모델을 결합한 기술입니다. 이 방법은 질문에 대한 답변을 생성할 때, 외부 데이터베이스에서 관련 정보를 찾아 활용합니다. 이렇게 하면 더 정확하고 풍부한 답변을 제공할 수 있습니다. RAG는 주로 자연어 처리 분야에서 사용되며, 챗봇이나 정보 검색 시스템에 유용합니다.


**system / human 메시지 구조**

실제 chat application에서는 역할을 나누는 것이 중요.

- system: 모델의 역할, 제약, 규칙
- human: 실제 사용자 질문

이 구조를 사용하면 "문서에 없는 내용은 말하지 마라", "항상 표 형태로 답하라" 같은 규칙을 명확히 줄 수 있다.

In [ ]:
chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 친절하고 정확한 AI 튜터다."),
        ("human", "{topic}를 {level} 수준에 맞게 {n_sentences}문장으로 설명해줘.")
    ]
)

msg = chat_prompt.invoke(
    {
        "topic": "LangChain",
        "level": "비전공 초보자",
        "n_sentences": 5
    }
)

print(llm.invoke(msg).content)

LangChain은 자연어 처리(NLP)와 관련된 작업을 쉽게 할 수 있도록 도와주는 도구입니다. 이 도구는 다양한 언어 모델과 연결하여 텍스트를 생성하거나 질문에 답하는 기능을 제공합니다. 사용자는 복잡한 코드를 작성하지 않고도 언어 모델을 활용할 수 있습니다. LangChain은 여러 기능을 모듈화하여 필요한 부분만 선택해 사용할 수 있게 설계되었습니다. 따라서 비전공자도 쉽게 접근하고 활용할 수 있는 장점이 있습니다.


**Chain**

LangChain의 중요한 아이디어는 여러 구성요소를 "흐름"으로 연결하는 것.

예를 들어:

입력 변수
→ prompt 생성
→ model 호출
→ 출력 파싱

이러한 흐름을 LangChain에서는 체인으로 다룰 수 있다.

---

## 핵심 문법

`|`

In [ ]:
chain = chat_prompt | llm
result = chain.invoke(
    {
        "topic": "Vector Store",
        "level": "학부생",
        "n_sentences": 4
    }
)

print(result.content)

Vector Store는 데이터를 벡터 형태로 저장하고 관리하는 시스템입니다. 벡터는 수치로 표현된 데이터 포인트로, 주로 기계 학습과 자연어 처리에서 사용됩니다. 이 저장소는 유사한 데이터 간의 거리 계산을 통해 빠르게 검색할 수 있도록 도와줍니다. 예를 들어, 텍스트 문서나 이미지의 특징을 벡터로 변환하여, 비슷한 항목을 쉽게 찾을 수 있게 합니다.


`chat_prompt | llm`

이 코드는

- 왼쪽에서 prompt를 만들고
- 그 결과를 오른쪽 모델의 입력으로 전달하는

연결 구조를 의미합니다.

**Output Parser**

모델 응답은 메시지 객체 형태일 수 있다.  

하지만 실제 앱에서는 문자열만 필요한 경우가 많음.

이때 Output Parser를 사용.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = chat_prompt | llm | StrOutputParser()

result = chain.invoke(
    {
        "topic": "Embedding",
        "level": "처음 배우는 사람",
        "n_sentences": 4
    }
)

print(result)
print(type(result))

Embedding은 단어, 문장, 또는 개체를 고차원 공간의 벡터로 변환하는 방법입니다. 이렇게 변환된 벡터는 의미적으로 유사한 것끼리 가까운 위치에 배치됩니다. 예를 들어, "왕"과 "여왕"은 비슷한 의미를 가지므로 벡터 공간에서 가까운 거리에 위치하게 됩니다. 이를 통해 컴퓨터가 자연어를 이해하고 처리하는 데 도움을 줍니다.
<class 'langchain_core.messages.base.TextAccessor'>


- Model: `ChatOpenAI`
- Prompt: `ChatPromptTemplate`
- Chain: `prompt | model`
- Output Parser: `StrOutputParser`

LangChain은 단순한 모델 호출 코드가 아니라 **입력 → 처리 → 출력**의 흐름을 구성하는 프레임워크라는 점.

**Document**

LangChain에서 문서는 단순 문자열이 아니라 메타데이터를 함께 가질 수 있는 객체로 다뤄짐.

- PDF에서 읽은 문서
- 웹 문서
- DB에서 가져온 텍스트
- 검색된 chunk

를 일관되게 다루는 데 중요.

In [ ]:
from langchain_core.documents import Document

docs = [
    Document(page_content="LangChain은 LLM 애플리케이션을 구성하기 위한 프레임워크이다.", metadata={"source": "doc1"}),
    Document(page_content="RAG는 검색된 문서를 바탕으로 답변을 생성하는 방식이다.", metadata={"source": "doc2"}),
    Document(page_content="Prompt template는 재사용 가능한 프롬프트를 만든다.", metadata={"source": "doc3"})
]

for d in docs:
    print(d)
    print("-"*60)

page_content='LangChain은 LLM 애플리케이션을 구성하기 위한 프레임워크이다.' metadata={'source': 'doc1'}
------------------------------------------------------------
page_content='RAG는 검색된 문서를 바탕으로 답변을 생성하는 방식이다.' metadata={'source': 'doc2'}
------------------------------------------------------------
page_content='Prompt template는 재사용 가능한 프롬프트를 만든다.' metadata={'source': 'doc3'}
------------------------------------------------------------


**문서를 context로 넣어 답변하게 만들기**

RAG를 배우기 전에 먼저 이해해야 할 것은 "문서를 모델 입력에 포함시키는 방식".

아직 검색은 하지 않고, 이미 준비된 문서들을 그대로 context로 넣어보자.

In [ ]:
context = "\n".join([doc.page_content for doc in docs])
print(context)

LangChain은 LLM 애플리케이션을 구성하기 위한 프레임워크이다.
RAG는 검색된 문서를 바탕으로 답변을 생성하는 방식이다.
Prompt template는 재사용 가능한 프롬프트를 만든다.


In [ ]:
context_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 주어진 문맥만 바탕으로 답하는 조교다. 문맥에 없으면 모른다고 답해라."),
        ("human", "문맥:\n{context}\n\n질문: {question}")
    ]
)

context_chain = context_prompt | llm | StrOutputParser()

print(
    context_chain.invoke(
        {
            "context": context,
            "question": "RAG는 무엇인가?"
        }
    )
)

RAG는 검색된 문서를 바탕으로 답변을 생성하는 방식이다.


In [ ]:
print(
    context_chain.invoke(
        {
            "context": context,
            "question": "LangGraph의 durable execution은 무엇인가?"
        }
    )
)

모른다.


이 단계는 아직 검색이 없기 때문에 완전한 RAG는 님.

하지만 이미 중요한 구조를 배움.

- 문서를 context로 넣고
- 모델이 그 문맥에 근거해 답하게 한다

**RAG의 생성 단계**를 먼저 이해한 것.

**왜 문서를 나누는가**

긴 문서를 한 번에 모델에 넣는 것은 비효율적입니다.

문서를 잘게 나누는 이유는 두 가지입니다.

1. 모델 입력 길이 제한
2. 관련 부분만 더 정확하게 검색하기 위해

이때 사용하는 것이 Text Splitter입니다.

In [ ]:
long_text = """
LangChain은 LLM 애플리케이션을 만들기 위한 프레임워크이다.
모델 호출, 프롬프트, 문서 로딩, 검색, 에이전트 같은 구성요소를 연결할 수 있다.
RAG는 Retrieval-Augmented Generation의 약자이다.
질문과 관련된 문서를 먼저 찾고, 그 문서를 바탕으로 답변을 생성한다.
텍스트를 chunk로 분할하는 이유는 검색 성능과 문맥 길이 제한 때문이다.
너무 긴 문서를 한 번에 넣으면 비효율적이고 관련 부분만 찾기 어렵다.
"""

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=80,
    chunk_overlap=20
)

chunks = splitter.split_text(long_text)

for i, chunk in enumerate(chunks, 1):
    print(f"[Chunk {i}]")
    print(chunk)
    print("-"*50)

[Chunk 1]
LangChain은 LLM 애플리케이션을 만들기 위한 프레임워크이다.
--------------------------------------------------
[Chunk 2]
모델 호출, 프롬프트, 문서 로딩, 검색, 에이전트 같은 구성요소를 연결할 수 있다.
--------------------------------------------------
[Chunk 3]
RAG는 Retrieval-Augmented Generation의 약자이다.
--------------------------------------------------
[Chunk 4]
질문과 관련된 문서를 먼저 찾고, 그 문서를 바탕으로 답변을 생성한다.
--------------------------------------------------
[Chunk 5]
텍스트를 chunk로 분할하는 이유는 검색 성능과 문맥 길이 제한 때문이다.
--------------------------------------------------
[Chunk 6]
너무 긴 문서를 한 번에 넣으면 비효율적이고 관련 부분만 찾기 어렵다.
--------------------------------------------------


In [ ]:
doc = Document(page_content=long_text)
split_docs = splitter.split_documents([doc])

for i, d in enumerate(split_docs, 1):
    print(f"[Doc Chunk {i}]")
    print(d.page_content)
    print("-"*50)

[Doc Chunk 1]
LangChain은 LLM 애플리케이션을 만들기 위한 프레임워크이다.
--------------------------------------------------
[Doc Chunk 2]
모델 호출, 프롬프트, 문서 로딩, 검색, 에이전트 같은 구성요소를 연결할 수 있다.
--------------------------------------------------
[Doc Chunk 3]
RAG는 Retrieval-Augmented Generation의 약자이다.
--------------------------------------------------
[Doc Chunk 4]
질문과 관련된 문서를 먼저 찾고, 그 문서를 바탕으로 답변을 생성한다.
--------------------------------------------------
[Doc Chunk 5]
텍스트를 chunk로 분할하는 이유는 검색 성능과 문맥 길이 제한 때문이다.
--------------------------------------------------
[Doc Chunk 6]
너무 긴 문서를 한 번에 넣으면 비효율적이고 관련 부분만 찾기 어렵다.
--------------------------------------------------


- 문서 분할은 Langchain의 RAG 인덱싱 흐름의 핵심 단계

*ㅕ*Vector Store와 Retriever**

문서를 잘게 나눈 뒤에는 질문과 관련된 chunk를 찾을 수 있어야 함.

이를 위해 보통 다음 단계를 거침.

1. 각 chunk를 embedding으로 변환
2. vector store에 저장
3. 질문이 들어오면 비슷한 chunk 검색

여기서 검색 역할을 하는 인터페이스가 Retriever.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [ ]:
retrieved_docs = retriever.invoke("왜 텍스트를 chunk로 나누나요?")

for i, d in enumerate(retrieved_docs, 1):
    print(f"[Retrieved {i}]")
    print(d.page_content)
    print("-"*50)

[Retrieved 1]
텍스트를 chunk로 분할하는 이유는 검색 성능과 문맥 길이 제한 때문이다.
--------------------------------------------------
[Retrieved 2]
너무 긴 문서를 한 번에 넣으면 비효율적이고 관련 부분만 찾기 어렵다.
--------------------------------------------------


**미니 RAG 구현**


질문

→ retriever가 관련 문서 검색

→ 검색 결과를 context로 조합

→ prompt에 context와 question 삽입

→ model이 최종 답변 생성

이것이 가장 기본적인 RAG 흐름.

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 문맥에 근거해서만 답하는 조교다. 문맥에 없으면 모른다고 답해라."),
        ("human", "문맥:\n{context}\n\n질문: {question}")
    ]
)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
answer = rag_chain.invoke("텍스트를 왜 분할하나요?")
print(answer)

텍스트를 분할하는 이유는 검색 성능과 문맥 길이 제한 때문입니다. 너무 긴 문서를 한 번에 넣으면 비효율적이고 관련 부분을 찾기 어렵기 때문입니다.


In [ ]:
answer = rag_chain.invoke("LangGraph의 human-in-the-loop는 무엇인가요?")
print(answer)

문맥에 LangGraph에 대한 정보가 없으므로, 그에 대한 답변을 드릴 수 없습니다.


In [ ]:
!pip install pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 2.0 MB/s eta 0:00:00


In [ ]:
# '황순원 작가 - [소나기]' 데이터 다운받기
!gdown --id 1lUaTmMHeEgINNESSDlWP7kitW-wvjOpZ

/usr/local/lib/python3.10/dist-packages/gdown/__main__.py:132: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1lUaTmMHeEgINNESSDlWP7kitW-wvjOpZ
To: /content/소나기 - 황순원.pdf
100% 124k/124k [00:00<00:00, 62.1MB/s]


#### 1. 데이터 로드
`PyPDFLoader`를 활용하여 PDF 파일을 로드합니다.

In [ ]:
from langchain.document_loaders import PyPDFLoader

# PDF 파일 로드
loader = PyPDFLoader("/content/소나기 - 황순원.pdf")
document = loader.load()

In [ ]:
document

[Document(metadata={'source': '/content/소나기 - 황순원.pdf', 'page': 0}, page_content='- 1 -소나기\n황순원\n소년은 개울가에서 소녀를 보자 곧 윤 초시네 증손녀 (曾孫女 )딸이라는 걸 알 수 있었다 . \n소녀는 개울에다 손을 잠그고 물장난을 하고 있는 것이다 . 서울서는 이런 개울물을 보지 \n못하기나 한 듯이.\n벌써 며칠째 소녀는 , 학교에서 돌아오는 길에 물장난이었다 . 그런데 , 어제까지 개울 기슭에\n서 하더니 , 오늘은 징검다리 한가운데 앉아서 하고 있다.\n소년은 개울둑에 앉아 버렸다 . 소녀가 비키기를 기다리자는 것이다 .\n요행 지나가는 사람이 있어, 소녀가 길을 비켜 주었다 .\n다음 날은 좀 늦게 개울가로 나왔다 .\n이 날은 소녀가 징검다리 한가운데 앉아 세수를 하고 있었다 . 분홍 스웨터 소매를 걷어올\n린 목덜미가 마냥 희었다 .\n한참 세수를 하고 나더니 , 이번에는 물 속을 빤히 들여다 본다. 얼굴이라도 비추어 보는 \n것이리라 . 갑자기 물을 움켜 낸다. 고기 새끼라도 지나가는 듯.\n소녀는 소년이 개울둑에 앉아 있는 걸 아는지 모르는지 그냥 날쌔게 물만 움켜 낸다. 그러\n나, 번번이 허탕이다 . 그대로 재미있는 양, 자꾸 물만 움킨다 . 어제처럼 개울을 건너는 사\n람이 있어야 길을 비킬 모양이다 .\n그러다가 소녀가 물 속에서 무엇을 하나 집어 낸다. 하얀 조약돌이었다 . 그리고는 벌떡 일\n어나 팔짝팔짝 징검다리를 뛰어 건너간다 .\n다 건너가더니만 홱 이리로 돌아서며 ,\n“이 바보.”\n조약돌이 날아왔다 .\n소년은 저도 모르게 벌떡 일어섰다 .\n단발 머리를 나풀거리며 소녀가 막 달린다 . 갈밭 사잇길로 들어섰다 . 뒤에는 청량한 가을 \n햇살 아래 빛나는 갈꽃뿐 .\n이제 저쯤 갈밭머리로 소녀가 나타나리라 . 꽤 오랜 시간이 지났다고 생각됐다 . 그런데도 \n소녀는 나타나지 않는다 . 발돋움을 했다. 그러고도 상당한 시간이 지났다고 생

In [ ]:
document[0].page_content[:100]

'- 1 -소나기\n황순원\n소년은 개울가에서 소녀를 보자 곧 윤 초시네 증손녀 (曾孫女 )딸이라는 걸 알 수 있었다 . \n소녀는 개울에다 손을 잠그고 물장난을 하고 있는 것이다 . 서'

#### 2. 데이터 분할

- TextSpliter : https://wikidocs.net/231430

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
texts = text_splitter.split_documents(document)

In [ ]:
texts[1].page_content

'갈꽃이 들길을 걸어가는 것만 같았다 .\n소년은 이 갈꽃이 아주 뵈지 않게 되기까지 그대로 서 있었다 . 문득, 소녀가 던지 조약돌\n을 내려다보았다 . 물기가 걷혀 있었다 . 소년은 조약돌을 집어 주머니에 넣었다 .\n다음 날부터 좀더 늦게 개울가로 나왔다 . 소녀의 그림자가 뵈지 않았다 . 다행이었다 .\n그러나 , 이상한 일이었다 . 소녀의 그림자가 뵈지 않는 날이 계속될수록 소년의 가슴 한 구\n석에는 어딘가 허전함이 자리 잡는 것이었다 . 주머니 속 조약돌을 주무르는 버릇이 생겼\n다.\n그러한 어떤 날, 소년은 전에 소녀가 앉아 물장난을 하던 징검다리 한가운데에 앉아 보았\n다. 물 속에 손을 잠갔다 . 세수를 하였다 . 물 속을 들여다보았다 . 검게 탄 얼굴이 그대로 \n비치었다 . 싫었다 .'

In [ ]:
for i in range(len(texts)):
    print(i, '번 인덱스 길이: ', len(texts[i].page_content))

0 번 인덱스 길이:  993
1 번 인덱스 길이:  390
2 번 인덱스 길이:  993
3 번 인덱스 길이:  351
4 번 인덱스 길이:  967
5 번 인덱스 길이:  164
6 번 인덱스 길이:  991
7 번 인덱스 길이:  567
8 번 인덱스 길이:  961
9 번 인덱스 길이:  311
10 번 인덱스 길이:  960
11 번 인덱스 길이:  558
12 번 인덱스 길이:  457


### 3. 저장 및 검색
OpenAIEmbeddings 를 활용하여 문서의 내용을 임베딩한 뒤, Chroma 벡터스토어(vectorstore) 에 저장합니다.

VectorStore는 자연어 --> 숫자 처리한 후 이들을 저장하는 벡터 저장소입니다.

벡터 저장소는 임베딩된 데이터를 인덱싱하여, input으로 받아들이는 query와의 유사도를 빠르게 출력합니다.

대표적으로 FAISS, Chroma가 존재합니다.

In [ ]:
!pip install openai chromadb tiktoken sentence_transformers

#### 3-1. 텍스트 임베딩

##### OpenAIEmbeddings - ada-002

In [ ]:
from langchain.embeddings import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings()

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  warn_deprecated(


In [ ]:
embeddings_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7fa040d0be80>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7fa040d09c60>, model='text-embedding-ada-002', deployment='text-embedding-ada-002', openai_api_version='', openai_api_base=None, openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key='sk-proj-kKo1J6euVSMIP49MS1MuT3BlbkFJQKKxPQJtDztuwlKEfIRi', openai_organization=None, allowed_special=set(), disallowed_special='all', chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None)

In [ ]:
embeddings = embeddings_model.embed_documents(
    [
        "안녕하세요",
        "제 이름은 홍길동입니다.",
        "이름이 무엇인가요?",
        "랭체인은 유용합니다.",
        "Hello World!"
    ]
)
len(embeddings), len(embeddings[0])

(5, 1536)

In [ ]:
embedded_query_q = embeddings_model.embed_query("이 대화에서 언급된 이름은 무엇입니까?")
embedded_query_a = embeddings_model.embed_query("이 대화에서 언급된 이름은 홍길동입니다.")
print(len(embedded_query_q), len(embedded_query_a))

1536 1536


In [ ]:
from numpy import dot
from numpy.linalg import norm
import numpy as np

def cos_sim(A, B):
       return dot(A, B)/(norm(A)*norm(B))

In [ ]:
print(cos_sim(embedded_query_q, embedded_query_a))
print(cos_sim(embedded_query_q, embeddings[1]))
print(cos_sim(embedded_query_q, embeddings[3]))

0.9012151416840555
0.8499725111377555
0.7753900276853167


##### Huggingface Embedding

In [ ]:
!pip install sentence_transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 1.2 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl (176.2 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-m

In [ ]:
from langchain.embeddings import HuggingFaceBgeEmbeddings

model_name = "BAAI/bge-small-en"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True}
hf = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/90.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings = hf.embed_documents(
    [
    "today is monday",
    "weather is nice today",
    "what's the problem?",
    "langhcain in useful",
    "Hello World!",
    "my name is morris"
    ]
)

In [ ]:
BGE_query_q = hf.embed_query("Hello? who is this?")
BGE_query_a = hf.embed_query("hi this is harrison")

print(cos_sim(BGE_query_q, BGE_query_a))
print(cos_sim(BGE_query_q, embeddings[1]))
print(cos_sim(BGE_query_q, embeddings[5]))

0.8522539963959582
0.7469068258795019
0.792870413704442


In [ ]:
sentences = [
    "안녕하세요",
    "제 이름은 홍길동입니다.",
    "이름이 무엇인가요?",
    "랭체인은 유용합니다.",
    "홍길동 아버지의 이름은 홍상직입니다."
    ]
ko_embeddings = hf.embed_documents(sentences)

In [ ]:
BGE_query_q_2 = hf.embed_query("홍길동은 아버지를 아버지라 부르지 못하였습니다. 홍길동 아버지의 이름은 무엇입니까?")
BGE_query_a_2 = hf.embed_query("홍길동의 아버지는 엄했습니다.")


print("질문: 홍길동은 아버지를 아버지라 부르지 못하였습니다. 홍길동 아버지의 이름은 무엇입니까? \n", "-"*100)
print("홍길동의 아버지는 엄했습니다. \t\t 문장 유사도: ", round(cos_sim(BGE_query_q_2, BGE_query_a_2),2))
print(sentences[1] + "\t\t\t 문장 유사도: ", round(cos_sim(BGE_query_q_2, ko_embeddings[1]),2))
print(sentences[3] + "\t\t\t 문장 유사도: ", round(cos_sim(BGE_query_q_2, ko_embeddings[3]),2))
print(sentences[4] + "\t 문장 유사도: ", round(cos_sim(BGE_query_q_2, ko_embeddings[4]),2))

질문: 홍길동은 아버지를 아버지라 부르지 못하였습니다. 홍길동 아버지의 이름은 무엇입니까? 
 ----------------------------------------------------------------------------------------------------
홍길동의 아버지는 엄했습니다. 		 문장 유사도:  0.95
제 이름은 홍길동입니다.			 문장 유사도:  0.88
랭체인은 유용합니다.			 문장 유사도:  0.85
홍길동 아버지의 이름은 홍상직입니다.	 문장 유사도:  0.92


##### 한국어 사전학습 모델 임베딩 - ko-sbert-nli

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

model_name = "jhgan/ko-sbert-nli"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True}
ko = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/538 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
sentences = [
    "안녕하세요",
    "제 이름은 홍길동입니다.",
    "이름이 무엇인가요?",
    "랭체인은 유용합니다.",
    "홍길동 아버지의 이름은 홍상직입니다."
    ]

ko_embeddings = ko.embed_documents(sentences)

q = "홍길동은 아버지를 아버지라 부르지 못하였습니다. 홍길동 아버지의 이름은 무엇입니까?"
a = "홍길동의 아버지는 엄했습니다."
ko_query_q = ko.embed_query(q)
ko_query_a = ko.embed_query(a)

print("질문: {} \n".format(q), "-"*100)
print("{} \t\t 문장 유사도: ".format(a), round(cos_sim(ko_query_q, ko_query_a),2))
print("{}\t\t\t 문장 유사도: ".format(sentences[1]), round(cos_sim(ko_query_q, ko_embeddings[1]),2))
print("{}\t\t\t 문장 유사도: ".format(sentences[3]), round(cos_sim(ko_query_q, ko_embeddings[3]),2))
print("{}\t 문장 유사도: ".format(sentences[4]), round(cos_sim(ko_query_q, ko_embeddings[4]),2))

질문: 홍길동은 아버지를 아버지라 부르지 못하였습니다. 홍길동 아버지의 이름은 무엇입니까? 
 ----------------------------------------------------------------------------------------------------
홍길동의 아버지는 엄했습니다. 		 문장 유사도:  0.47
제 이름은 홍길동입니다.			 문장 유사도:  0.54
랭체인은 유용합니다.			 문장 유사도:  0.03
홍길동 아버지의 이름은 홍상직입니다.	 문장 유사도:  0.61


#### 3-2. Vector Store
- https://wikidocs.net/231578

In [ ]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.4/581.4 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 56.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.0/107.0 kB 11.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 7.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.7/283.7 kB 29.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 6

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

sentences = [
    "안녕하세요",
    "제 이름은 홍길동입니다.",
    "이름이 무엇인가요?",
    "랭체인은 유용합니다.",
    "홍길동 아버지의 이름은 홍상직입니다."
    ]

embeddings_model = OpenAIEmbeddings()
db = Chroma.from_texts(
    sentences,
    embeddings_model,
    collection_name = 'history',
    persist_directory = './db/chromadb',
    collection_metadata = {'hnsw:space': 'cosine'}, # l2 is the default
)

db

query 변수에 검색 쿼리를 정의합니다.
db.similarity_search 메소드를 사용하여 저장된 데이터 중에서 쿼리와 가장 유사한 문서를 찾습니다.
검색 결과를 docs 변수에 저장하고, 가장 유사한 문서의 내용은 docs[0].page_content를 통해 확인합니다.

In [ ]:
query = '홍길동은 아버지를 아버지라 부르지 못하였습니다. 홍길동 아버지의 이름은 무엇입니까?'
docs = db.similarity_search(query)
print(docs[0].page_content)

홍길동 아버지의 이름은 홍상직입니다.


#### 3-3. Retriever
리트리버는 비정형 쿼리가 주어지면 문서를 반환하는 인터페이스입니다. 리트리버는 문서를 저장할 필요 없이 단지 반환(또는 검색)만 할 수 있습니다. - Langchain Document

쉽게 말해, Retriever는 검색을 쉽게 할 수 있도록 구성된 모듈입니다. 이를 통해 손쉽게 문서를 검색할 수 있도록 하고 이를 기반으로 LLM과 대화할 수 있도록 합니다.


In [ ]:
# as_retriever 메소드를 사용하여 벡터스토어에서 Retriever 객체를 생성합니다
retriever = db.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7f9f2086cbb0>)

In [ ]:
# 검색 쿼리
query = '홍길동 아버지의 이름은 무엇입니까?'

# 가장 유사도가 높은 문장을 추출
retriever.get_relevant_documents(query)

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


[Document(page_content='홍길동 아버지의 이름은 홍상직입니다.'),
 Document(page_content='제 이름은 홍길동입니다.'),
 Document(page_content='이름이 무엇인가요?'),
 Document(page_content='안녕하세요')]

In [ ]:
# 가장 유사도가 높은 문장을 하나만 추출
retriever = db.as_retriever(search_kwargs={'k': 1})
retriever.get_relevant_documents(query)

[Document(page_content='홍길동 아버지의 이름은 홍상직입니다.')]

### RAG 프롬프트 템플릿


-  langchain hub 에서 Prompt 다운로드 예시(https://smith.langchain.com/hub)
- https://smith.langchain.com/hub/rlm/rag-prompt

In [ ]:
!pip install langchainhub

In [ ]:
from langchain import hub

rag_prompt = hub.pull("rlm/rag-prompt")
rag_prompt

ChatPromptTemplate(input_variables=['context', 'question'], metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"))])

In [ ]:
rag_prompt.format(context="홍길동의 아버지는 홍상직입니다.", question="홍길동의 아버지는 누구입니까?")

"Human: You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: 홍길동의 아버지는 누구입니까? \nContext: 홍길동의 아버지는 홍상직입니다. \nAnswer:"

### 체인 생성

LLM 모델을 정의하고 Chain을 생성해봅시다.
- chain 생성 설명 : https://wikidocs.net/233344
- Runnable 설명 : https://wikidocs.net/233346

- 코드 예시 : https://python.langchain.com/docs/expression_language/primitives/passthrough/

In [ ]:
# LLM
from langchain.chat_models import ChatOpenAI

# ChatGPT 모델 지정
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

In [ ]:
# RAG chain 생성
from langchain.schema.runnable import RunnablePassthrough

# pipe operator를 활용한 체인 생성
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
)

In [ ]:
rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7f9f2086cbb0>, search_kwargs={'k': 1}),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"))])
| ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x7f9f22961360>, async_client=<openai.resources.chat.completions.AsyncCompletions o

In [ ]:
rag_chain.invoke("이 소설의 제목은 뭐야?")


AIMessage(content='제목을 알 수 없습니다.', response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 95, 'total_tokens': 104}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-36f69a37-ee86-45d8-bff2-c1815245af08-0')

In [ ]:
rag_chain.invoke("소녀는 어떤 옷을 입고 있어?")


AIMessage(content='저는 그 질문에 대한 정보가 없습니다.', response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 100, 'total_tokens': 115}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-66a6bb82-1787-45ea-b41a-23bd31d08df7-0')

In [ ]:
rag_chain.invoke("이 소설의 결말이 뭐야?")


AIMessage(content='제가 죄송합니다. 이 소설의 결말에 대한 정보를 찾을 수 없습니다.', response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 96, 'total_tokens': 126}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-43133555-3dd0-441f-a32d-3c704e7cebb2-0')

### 실습

In [ ]:
# 데이터 로드
loader = PyPDFLoader("/content/소나기 - 황순원.pdf")
document = loader.load()

In [ ]:
# 데이터 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
texts = text_splitter.split_documents(document)

In [ ]:
# 임베딩
embeddings = OpenAIEmbeddings()

# Chroma DB 에 저장
docsearch = Chroma.from_documents(texts, embeddings)

# retriever 가져옴
retriever = docsearch.as_retriever()

In [ ]:
# rag 프롬프트 다운로드
rag_prompt = hub.pull("rlm/rag-prompt")


In [ ]:
# ChatGPT 모델 지정
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

# pipe operator를 활용한 체인 생성
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
)

In [ ]:
rag_chain.invoke("소녀는 어떤 옷을 입고 있어?")


AIMessage(content='분홍 스웨터를 입고 있어.', response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 4112, 'total_tokens': 4126}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-e7c88cb9-a023-4d13-9f99-481947d8cb40-0')

## 문제 : 네이버 뉴스를 바탕으로 Q&A 어플리케이션을 만들려고 합니다. RAG를 활용하여 만들어보세요.
- 네이버 기사 : https://www.wowtv.co.kr/NewsCenter/News/Read?articleId=A202404050144&t=NN

In [ ]:
# 힌트 : URL Document Loader를 사용하세요